# Structured Outputs and Tools

This notebook shows how to keep outputs typed and tool contracts stable so downstream code can trust them.


## Why structured output matters

If your agent returns sections or JSON, you can validate the result before it reaches another system. That is much safer than free-form text.


In [ ]:
from dataclasses import dataclass

@dataclass
class ActionPlan:
    summary: str
    priority: str
    owner: str
    citations: list[str]

plan = ActionPlan(
    summary='Investigate auth latency around shift change',
    priority='high',
    owner='identity',
    citations=['doc-21', 'doc-44'],
)
plan


## Tool contract tip

A tool should have a stable input and a stable output. You can change the implementation later without changing the contract.


In [ ]:
def search_knowledge_base(query: str) -> dict:
    return {
        'query': query,
        'results': [
            {'id': 'kb-1', 'score': 0.91, 'title': 'Auth timeout troubleshooting'},
            {'id': 'kb-2', 'score': 0.74, 'title': 'Cache saturation symptoms'},
        ],
    }

search_knowledge_base('authentication timeout after shift change')


## Modern workflow trick

Use tools for facts, then use the model for reasoning and presentation. Do not mix the two if you can avoid it.


In [ ]:
def build_final_payload(summary: ActionPlan, kb_result: dict) -> dict:
    return {
        'summary': summary.summary,
        'priority': summary.priority,
        'owner': summary.owner,
        'evidence': [item['id'] for item in kb_result['results']],
        'next_step': 'assign to the owner and confirm impact',
    }

build_final_payload(plan, search_knowledge_base('auth timeout'))


## Tip

If the output will be consumed by code, validate it like code. If it will be consumed by a person, still make it structured enough to trust.
